# Fine-tuning T5-flan Model using LoRA

Parameter-Efficient Fine-Tuning (PEFT). PEFT techniques are the "cheat codes" of the LLM world, allowing us to adapt massive models on consumer hardware. The reigning champion of these techniques is LoRA (Low-Rank Adaptation).

In this notebook, we are going to demystify LoRA and show you exactly how to use it.





The Code: Summarizing Dialogue with FLAN-T5
Let's get our hands dirty. We will fine-tune Google's flan-t5-small model on the SAMSum dataset to learn how to summarize casual conversations.

Prerequisites: Install necessary libraries (transformers, peft, datasets, evaluate, rouge_score, accelerate).

In [ ]:
# !pip install -U --no-cache-dir accelerate peft bitsandbytes transformers trl

In [ ]:
# Import the libraries
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    TrainingArguments, 
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import (
    PeftModel,
    LoraConfig, 
    get_peft_model, 
    TaskType, 
    prepare_model_for_kbit_training
)
import random
import warnings
# Filter out syntax warnings from libraries
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
# Verify GPU is active
if not torch.cuda.is_available():
    raise ValueError("❌ GPU not detected! Check your Kaggle settings.")
print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

You can use Kaggle GPU for this task. On your right, under session options, under accelerator

In [ ]:
# 1. Setup Model and Tokenizer
model_name = "google/flan-t5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# T5 is an Encoder-Decoder model, so we load it as such
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)
model.config.tie_word_embeddings = False

Since it is a summarization task, we are using T5, which is an "Encoder-Decoder" (Seq2Seq) model. This is different from GPT-style models (Decoder-only). It reads text in one block and generates text in another.

In [ ]:
# 2. Prepare Data (Seq2Seq specific formatting)
# T5 needs inputs (Encoder) and targets (Decoder) separately.
data_files = {
    'train': '/kaggle/input/samsum-dataset-for-chat-summarization/train.json',
    'test': '/kaggle/input/samsum-dataset-for-chat-summarization/test.json'
}
dataset = load_dataset('json', data_files=data_files)

def preprocess_function(examples):
    # Add a prompt so the model knows what to do
    inputs = [f"Summarize this conversation:\n\n{dialogue}" for dialogue in examples['dialogue']]
    
    # Tokenize inputs (the conversation)
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    # Tokenize targets (the summary)
    labels = tokenizer(text_target=examples['summary'], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=["id", "dialogue", "summary"])

# Data collator handles dynamic padding for Seq2Seq batches
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

T5 needs a specific input format. We need to structure our data as "Instruction + Input" for the source, and the "Summary" as the target label.

In [ ]:
# 3. Setup PEFT (LoRA)
# T5 requires TASK_TYPE="SEQ_2_SEQ_LM", not "CAUSAL_LM"
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.05,
    bias="none",
    # Which layers to apply LoRA to. For T5, these are the attention mechanisms.
    target_modules=["q", "v"]
)

# Wrap the base model with the LoRA configuration
model = get_peft_model(model, peft_config)

# Let's see how efficient we are!
model.print_trainable_parameters()
# Output indicates we are training < 1% of parameters!

In [ ]:
# 4. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="flan-t5-samsum-lora",
    learning_rate=2e-4, # lora likes slightly higher learning rate
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=5, # change it 1 for the first run to check if code is working
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,        # Print training loss every 10 steps (instead of 500)
    report_to="none",        # Print to the cell output (turns off WandB/Tensorboard)
    disable_tqdm=False,      # Force the progress bar to show
    # Optional: This enables generation during evaluation (good for summarization)
    predict_with_generate=True,
    dataloader_pin_memory=True
)

# 5. Trainer
# Use Seq2SeqTrainer instead of SFTTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,                  # Now passes the correct argument type
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,          # Updated from 'tokenizer'
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

# Start training!
trainer.train()

Because we are using a Seq2Seq model, we use Seq2SeqTrainingArguments and Seq2SeqTrainer.

Seq2SeqTrainingArguments allows us to use predict_with_generate=True, which means during evaluation, the model will actually generate text summaries that we can score using metrics like ROUGE, rather than just calculating loss.

# Inference pipeline

Testing the model using a random input from test dataset 

In [ ]:
# 1. Load the model you just trained
# (We use the model object currently in memory)
model.eval()

# 2. Pick a random sample from the test set

sample = dataset['test'][random.randint(0, len(dataset['test']))]
dialogue = sample['dialogue']
ground_truth = sample['summary']

# 3. Prepare input
input_text = f"Summarize the following conversation:\n\n{dialogue}"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

# 4. Generate Summary
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"], 
        max_new_tokens=100, 
        do_sample=True, 
        top_p=0.9
    )

print("-" * 50)
print(f"INPUT DIALOGUE:\n{dialogue}")
print("-" * 50)
print(f"ACTUAL SUMMARY (Label):\n{ground_truth}")
print("-" * 50)
print(f"YOUR MODEL'S SUMMARY:\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")
print("-" * 50)

In [ ]:
# Save the adapter and tokenizer
output_path = "flan-t5-samsum-lora-final"
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

print(f"✅ Model saved to: {output_path}")

Inferencing on a new example

In [ ]:
# 1. Setup paths
base_model_name = "google/flan-t5-small"
adapter_path = "flan-t5-samsum-lora-final" # This is the output_dir from the training step

# 2. Load the Base Model & Tokenizer
print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

# 3. Load the LoRA Adapter
# This overlays your trained weights onto the base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval() # Switch to evaluation mode

# 4. Define an Inference Function
def summarize_conversation(dialogue):
    # Format inputs just like we did during training
    input_text = f"Summarize the following conversation:\n\n{dialogue}"
    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    # Generate Summary
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"], 
            max_new_tokens=100, 
            do_sample=True, 
            top_p=0.9
        )
    
    # Decode Output
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

# 5. Run it on a new sample
sample_dialogue = """
John: Hey, are we still on for the meeting at 3 PM?
Sarah: I might be a few minutes late. Traffic is terrible.
John: No worries. Should we start without you or wait?
Sarah: Please start without me. I'll join as soon as I can.
John: Okay, see you soon.
"""

print("-" * 30)
print(f"INPUT DIALOGUE:\n{sample_dialogue}")
print("-" * 30)
print(f"GENERATED SUMMARY:\n{summarize_conversation(sample_dialogue)}")
print("-" * 30)

In [ ]:
# Zip the folder so it's easy to download
!zip -r flan_t5_lora.zip flan-t5-samsum-lora-final